In [1]:
# ============================================================
# STEP 3 — Tokenisation
# Loads the saved CSVs, tokenises with BERT, and builds
# PyTorch DataLoaders ready for training.
# ============================================================

import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from google.colab import drive

# ============================================================
# 3.0  Mount Drive + Folders
# ============================================================
drive.mount("/content/drive", force_remount=True)

SAVE_DIR   = "/content/drive/MyDrive/PhishingClassifier"
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 256
BATCH_SIZE = 16

for folder in [SAVE_DIR,
               f"{SAVE_DIR}/model",
               f"{SAVE_DIR}/logs",
               f"{SAVE_DIR}/plots"]:
    os.makedirs(folder, exist_ok=True)

# ============================================================
# 3.1  Check CSV Files Exist
# ============================================================
print("=" * 50)
print("CHECKING DATA FILES")
print("=" * 50)

for name in ["train.csv", "val.csv", "test.csv"]:
    path = f"{SAVE_DIR}/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        rows = pd.read_csv(path).shape[0]
        print(f"  OK : {name}  ({rows} rows)")
    else:
        raise FileNotFoundError(
            f"\n  MISSING: {name}\n"
            f"  Please run 02_data_preparation.py first and confirm "
            f"'STEP 2 COMPLETE' appears at the end."
        )

# ============================================================
# 3.2  Load + Clean Splits
# ============================================================
print("\n" + "=" * 50)
print("LOADING SPLITS")
print("=" * 50)

def load_split(path, name):
    df = pd.read_csv(path)
    df["text"]  = df["text"].astype(str).str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.len() > 0]
    df["text"]  = df["text"].fillna("empty")
    df["label"] = df["label"].astype(int)
    df = df.reset_index(drop=True)
    print(f"  {name}: {len(df)} rows  |  labels: {df['label'].value_counts().to_dict()}")
    return df

train_df = load_split(f"{SAVE_DIR}/train.csv", "Train")
val_df   = load_split(f"{SAVE_DIR}/val.csv",   "Val  ")
test_df  = load_split(f"{SAVE_DIR}/test.csv",  "Test ")

# ============================================================
# 3.3  Load Tokenizer
# ============================================================
print("\n" + "=" * 50)
print("LOADING TOKENIZER")
print("=" * 50)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  Model      : {MODEL_NAME}")
print(f"  Vocab size : {tokenizer.vocab_size:,}")
print(f"  Max length : {MAX_LEN}")

# Quick demo
sample = "Urgent! Your PayPal account is limited. Verify now at http://paypa1.ru"
tokens = tokenizer.tokenize(sample)
print(f"\n  Sample     : {sample}")
print(f"  Tokens     : {tokens}")
print(f"  Count      : {len(tokens)}")

# ============================================================
# 3.4  Dataset Class
# ============================================================
class PhishingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = [str(t).strip() if str(t).strip() else "empty"
                          for t in texts.tolist()]
        self.labels    = [int(l) for l in labels.tolist()]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ============================================================
# 3.5  Build DataLoaders
# ============================================================
print("\n" + "=" * 50)
print("BUILDING DATALOADERS")
print("=" * 50)

train_dataset = PhishingDataset(train_df["text"], train_df["label"], tokenizer, MAX_LEN)
val_dataset   = PhishingDataset(val_df["text"],   val_df["label"],   tokenizer, MAX_LEN)
test_dataset  = PhishingDataset(test_df["text"],  test_df["label"],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"  Train : {len(train_dataset):,} samples  →  {len(train_loader)} batches")
print(f"  Val   : {len(val_dataset):,} samples  →  {len(val_loader)} batches")
print(f"  Test  : {len(test_dataset):,} samples  →  {len(test_loader)} batches")

# ============================================================
# 3.6  Verify One Batch
# ============================================================
print("\n" + "=" * 50)
print("VERIFYING BATCH")
print("=" * 50)

batch = next(iter(train_loader))
print(f"  input_ids      : {batch['input_ids'].shape}   dtype={batch['input_ids'].dtype}")
print(f"  attention_mask : {batch['attention_mask'].shape}   dtype={batch['attention_mask'].dtype}")
print(f"  labels         : {batch['label'].shape}   dtype={batch['label'].dtype}")
print(f"  Label sample   : {batch['label'][:8].tolist()}")

print("\nStep 3 complete. Proceed to 04_model_training.py")

Mounted at /content/drive
CHECKING DATA FILES
  OK : train.csv  (19332 rows)
  OK : val.csv  (4143 rows)
  OK : test.csv  (4143 rows)

LOADING SPLITS
  Train: 19332 rows  |  labels: {1: 9666, 0: 9666}
  Val  : 4143 rows  |  labels: {1: 2072, 0: 2071}
  Test : 4143 rows  |  labels: {0: 2072, 1: 2071}

LOADING TOKENIZER


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Model      : bert-base-uncased
  Vocab size : 30,522
  Max length : 256

  Sample     : Urgent! Your PayPal account is limited. Verify now at http://paypa1.ru
  Tokens     : ['urgent', '!', 'your', 'pay', '##pal', 'account', 'is', 'limited', '.', 'verify', 'now', 'at', 'http', ':', '/', '/', 'pay', '##pa', '##1', '.', 'ru']
  Count      : 21

BUILDING DATALOADERS
  Train : 19,332 samples  →  1209 batches
  Val   : 4,143 samples  →  259 batches
  Test  : 4,143 samples  →  259 batches

VERIFYING BATCH


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  input_ids      : torch.Size([16, 256])   dtype=torch.int64
  attention_mask : torch.Size([16, 256])   dtype=torch.int64
  labels         : torch.Size([16])   dtype=torch.int64
  Label sample   : [0, 0, 1, 1, 1, 0, 1, 1]

Step 3 complete. Proceed to 04_model_training.py
